# Tema: Laboratorio integral e-commerce

## Objetivos
Resolver de forma autónoma un flujo con SQL, PySpark, Delta, CDC, streaming y Unity Catalog.

## Conceptos importantes para el examen
Selección de ingesta; calidad; Medallion; MERGE; deduplicación; joins; agregaciones; permisos; recuperación y operación.

**Dificultad:** Examen · **Tiempo:** 180–240 min.

No hay ejemplos guiados. Trabaja con las fuentes proporcionadas y toma tus propias decisiones. Ejecuta la preparación una vez. Para comparar con la referencia, inicia otra preparación completa y ejecuta las soluciones en orden en el nuevo schema. No ejecutes soluciones sobre tus resultados parciales sin revisar sus escrituras.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Escenario y contratos
Una tienda recibe snapshots de clientes/productos/pedidos, cambios de clientes y eventos JSON. Necesita ingresos por ciudad actual y día, además de actividad por tipo de evento. Los datos contienen duplicados, una cantidad negativa y un cliente desconocido. No elimines errores sin conservar evidencia.

La preparación crea únicamente fuentes ficticias y un volumen; necesita los permisos descritos en el README. Ninguna tabla Bronze/Silver/Gold está construida todavía.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_26_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.

# Requiere CREATE VOLUME en el schema; alternativa: usa un volumen autorizado.
spark.sql("CREATE VOLUME IF NOT EXISTS lab_files")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/lab_files"
dbutils.fs.mkdirs(BASE + "/landing")
CHECKPOINT = BASE + "/checkpoints/main"
print(BASE)
import json
customers_src = spark.createDataFrame([(i,f"Cliente {i:02d}",["Madrid","Bilbao","Sevilla"][i%3],datetime(2026,1,1)) for i in range(1,13)], "customer_id INT, name STRING, city STRING, updated_at TIMESTAMP")
products_src = spark.createDataFrame([(i,f"Producto {i}",float(i*10)) for i in range(1,7)], "product_id INT, product_name STRING, unit_price DOUBLE")
order_rows = [(i,(i-1)%12+1,(i-1)%6+1,str(i%3+1),"2026-01-10",datetime(2026,1,10),i) for i in range(1,25)]
order_rows += [order_rows[2],(25,1,1,"-1","2026-01-10",datetime(2026,1,10),25),(26,999,2,"1","2026-01-10",datetime(2026,1,10),26)]
orders_src = spark.createDataFrame(order_rows,"order_id INT, customer_id INT, product_id INT, quantity_raw STRING, order_date_raw STRING, updated_at TIMESTAMP, event_seq LONG")
customer_changes_src = spark.createDataFrame([
 (2,"Cliente 02","Valencia",datetime(2026,2,1),1),
 (2,"Cliente 02","Cádiz",datetime(2026,3,1),2),
 (5,"Cliente 05","Lugo",datetime(2026,2,1),3),
 (13,"Cliente 13","Madrid",datetime(2026,2,1),4)],
 "customer_id INT, name STRING, city STRING, updated_at TIMESTAMP, event_seq LONG")
for name,frame in [("source_customers",customers_src),("source_products",products_src),("source_orders",orders_src),("source_customer_changes",customer_changes_src)]:
    frame.write.format("delta").mode("overwrite").saveAsTable(name)
event_rows = [{"event_id":i,"customer_id":(i-1)%12+1,"event_type":"view" if i%2 else "cart","event_time":"2026-01-10T10:00:00Z"} for i in range(1,13)]
LANDING = BASE + "/landing"
dbutils.fs.put(LANDING + "/events_01.json","\n".join(json.dumps(x) for x in event_rows),overwrite=False)
print("Fuentes preparadas. Diseña tu solución antes de leer las referencias del final.")


## TAREAS — resuelve sin consultar las soluciones

### TAREA 1
Diseña la arquitectura: asigna herramienta y responsabilidad a cada entrada, define objetos Bronze/Silver/Gold y decide cómputo. Guarda tu decisión en un DataFrame.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 2
Inspecciona esquemas, recuentos y calidad de origen sin cambiar los datos. Identifica al menos tres problemas.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 3
Crea Bronze para customers, products, orders y cambios de clientes, conservando datos de origen y fecha de ingesta.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 4
Tipa pedidos y separa inválidos con motivo. No pierdas el valor raw.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 5
Deduplica los pedidos válidos con orden determinista y comprueba unicidad.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 6
Crea Silver de clientes/productos y activa captura de cambios de clientes antes de aplicar novedades.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 7
Resuelve claves huérfanas de pedidos sin perder evidencia. Publica Silver con solo pedidos válidos y dimensiones existentes.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 8
Construye Gold por día y ciudad con ingresos y número de pedidos; prueba que los joins no multiplican filas.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 9
Deduplica el CDC entrante y aplica SCD1 a clientes sin permitir retrocesos temporales. Deben quedar 13 clientes.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 10
Reenvía el mismo lote de clientes y demuestra idempotencia de datos. Recalcula Gold y explica por qué cambia la distribución por ciudad.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 11
Conserva en Delta la auditoría CDF posterior a la activación. Distingue preimage y postimage.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 12
Ingiere los eventos JSON incrementalmente en Bronze usando un checkpoint duradero; elige y justifica la fuente.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 13
Añade tres eventos nuevos (13–15), reinicia la consulta y vuelve a ejecutarla sin novedades; demuestra 15 filas.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 14
Publica Silver de eventos y Gold por tipo de evento; diferencia deduplicación de negocio y seguimiento del checkpoint.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 15
Implementa controles de publicación: unicidad, conservación de filas, ausencia de cantidades inválidas e ingresos esperados.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 16
Publica una vista Gold y define permisos mínimos para un lector. Ejecuta GRANT solo con autoridad y grupo existente.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 17
Propón un DAG desplegable y una estrategia de reintentos. Define cómo bloquearías Gold cuando falla calidad y qué métricas revisarías.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


### TAREA 18
Entrega evidencias finales y una decisión SCD1 vs SCD2. Consulta historial y documenta limitaciones de tu implementación.

In [ ]:
# ESCRIBE TU CÓDIGO Y JUSTIFICACIÓN AQUÍ


## Pistas opcionales
**1.** Separa copia de origen, calidad y consumo.

**2.** Busca duplicados, cantidades negativas y claves huérfanas.

**3.** Usa nombres propios del schema actual.

**4.** Convierte de forma tolerante y valida clave, fecha y cantidad positiva.

**5.** Ordena updated_at y event_seq; el duplicado exacto puede reducirse primero.

**6.** CDF debe estar habilitado antes del cambio que quieres observar.

**7.** Anti joins identifican lo que no casa; no descartes silenciosamente.

**8.** Precio × cantidad; usa ciudad actual para este informe.

**9.** El cliente 2 tiene dos cambios; elige el de marzo.

**10.** Conserva una instantánea antes del replay.

**11.** Lee desde CDF_START.

**12.** Para 12 JSON de contrato fijo, readStream.json con esquema explícito es suficiente; Auto Loader también es válido.

**13.** Nuevos nombres de archivo, mismo checkpoint y mismo destino.

**14.** El checkpoint no elimina dos eventos iguales presentes en archivos diferentes.

**15.** Separa inválidos, duplicados y huérfanos en el balance.

**16.** No concedas acceso a Bronze por conveniencia.

**17.** Extrae código de trabajo; no programes el notebook de ejercicios entero.

**18.** Relaciona cada requisito con consulta o comprobación, no solo con una frase.

## Preguntas de decisión

1. Dos ficheros distintos contienen el mismo order_id. ¿Qué evita doble contabilización?

A. Cambiar el checkpoint.

B. Deduplicar por clave y secuencia antes de Gold.

C. Usar force=true.

D. Aumentar el tamaño del cluster.

2. BI exige ciudad vigente en el momento de la compra. ¿Qué diseño corresponde?

A. SCD1 sin historial.

B. Una vista temporal sin dimensión.

C. SCD2 con join temporal.

D. COPY INTO sin transformaciones.

3. Una tarea de calidad falla. ¿Qué debe suceder antes de publicar Gold?

A. Corregir la causa y reparar las tareas afectadas.

B. Saltar Silver siempre.

C. Borrar los checkpoints de todos los flujos.

D. Dar ALL PRIVILEGES a todos.

## Reto adicional sin solución
Introduce una baja de cliente, una llegada tardía y una segunda versión del mismo pedido. Mantén resultados coherentes, documenta el tratamiento de compras históricas y demuestra que un replay no altera el resultado.

In [ ]:
# RETO ADICIONAL


## Autoevaluación
Cada tarea: 0 sin resolver, 1 funciona parcialmente/sin evidencia, 2 funciona con evidencia y decisión justificada. Máximo 36. Una puntuación alta aquí no equivale a una nota oficial. Revisa especialmente pérdida de registros, duplicados, uso de checkpoints y permisos excesivos.

---

## SOLUCIONES COMPLETAS — al final
No avances hasta terminar tu intento. Las alternativas válidas pueden diferir de esta referencia. Ejecuta preparación de nuevo y después estas soluciones en orden.

### Solución 1

In [ ]:
architecture = spark.createDataFrame([("customers/products/orders","batch Delta","Bronze de tablas fuente"),("customer changes","window + MERGE + CDF","Silver actual y auditoría"),("events JSON","Structured Streaming","Bronze incremental"),("métricas","SQL + Delta","Gold para BI")], "source STRING, technology STRING, purpose STRING")
display(architecture)
# Notebook PySpark: serverless o clásico compatible; BI: SQL warehouse.

### Solución 2

In [ ]:
for table in ["source_customers","source_products","source_orders","source_customer_changes"]:
    print(table,spark.table(table).count())
    spark.table(table).printSchema()
display(spark.table("source_orders").groupBy("order_id").count().filter("count>1"))
display(spark.table("source_orders").filter("try_cast(quantity_raw AS INT)<0 OR customer_id=999"))

### Solución 3

In [ ]:
for entity in ["customers","products","orders","customer_changes"]:
    (spark.table("source_"+entity).withColumn("ingested_at",F.current_timestamp())
     .write.format("delta").mode("overwrite").saveAsTable("bronze_"+entity))

### Solución 4

In [ ]:
typed_orders = (spark.table("bronze_orders").withColumn("quantity",F.expr("try_cast(quantity_raw AS INT)"))
 .withColumn("order_date",F.expr("try_cast(order_date_raw AS DATE)")))
valid_rule = "order_id IS NOT NULL AND quantity IS NOT NULL AND quantity>0 AND order_date IS NOT NULL"
invalid_orders = typed_orders.filter(f"NOT ({valid_rule})").withColumn("reject_reason",F.lit("invalid_key_quantity_or_date"))
invalid_orders.write.format("delta").mode("overwrite").saveAsTable("quarantine_invalid_orders")
valid_orders = typed_orders.filter(valid_rule)

### Solución 5

In [ ]:
w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc(),F.col("event_seq").desc())
dedup_orders = valid_orders.dropDuplicates().withColumn("rn",F.row_number().over(w)).filter("rn=1").drop("rn")
assert dedup_orders.count()==25

### Solución 6

In [ ]:
spark.sql("CREATE OR REPLACE TABLE silver_customers USING DELTA AS SELECT customer_id,name,city,updated_at FROM bronze_customers")
spark.sql("CREATE OR REPLACE TABLE silver_products USING DELTA AS SELECT product_id,product_name,unit_price FROM bronze_products")
spark.sql("ALTER TABLE silver_customers SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
CDF_START = spark.sql("DESCRIBE HISTORY silver_customers").agg(F.max("version")).first()[0]

### Solución 7

In [ ]:
customers_keys=spark.table("silver_customers").select("customer_id")
products_keys=spark.table("silver_products").select("product_id")
missing_customers=dedup_orders.join(customers_keys,"customer_id","left_anti").withColumn("reject_reason",F.lit("unknown_customer"))
known_customers=dedup_orders.join(customers_keys,"customer_id","inner")
missing_products=known_customers.join(products_keys,"product_id","left_anti").withColumn("reject_reason",F.lit("unknown_product"))
missing_customers.unionByName(missing_products).write.format("delta").mode("overwrite").saveAsTable("quarantine_orphan_orders")
known_customers.join(products_keys,"product_id","inner").write.format("delta").mode("overwrite").saveAsTable("silver_orders")
assert spark.table("silver_orders").count()==24

### Solución 8

In [ ]:
def refresh_gold():
    spark.sql("""CREATE OR REPLACE TABLE gold_daily_city USING DELTA AS
    SELECT o.order_date,c.city,COUNT(*) orders,SUM(o.quantity*p.unit_price) revenue
    FROM silver_orders o JOIN silver_customers c ON o.customer_id=c.customer_id
    JOIN silver_products p ON o.product_id=p.product_id GROUP BY o.order_date,c.city""")
refresh_gold()
assert spark.table("gold_daily_city").agg(F.sum("orders")).first()[0]==24
display(spark.table("gold_daily_city"))

### Solución 9

In [ ]:
w=Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc(),F.col("event_seq").desc())
latest=(spark.table("bronze_customer_changes").withColumn("rn",F.row_number().over(w)).filter("rn=1")
 .select("customer_id","name","city","updated_at"))
latest.createOrReplaceTempView("latest_customer_changes")
merge_sql="""MERGE INTO silver_customers t USING latest_customer_changes s ON t.customer_id=s.customer_id
WHEN MATCHED AND s.updated_at>t.updated_at THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *"""
spark.sql(merge_sql)
assert spark.table("silver_customers").count()==13
assert spark.table("silver_customers").filter("customer_id=2").first().city=="Cádiz"

### Solución 10

In [ ]:
spark.sql("CREATE OR REPLACE TABLE replay_before USING DELTA AS SELECT * FROM silver_customers")
spark.sql(merge_sql)
a,b=spark.table("replay_before"),spark.table("silver_customers")
assert a.exceptAll(b).count()==b.exceptAll(a).count()==0
refresh_gold()
# Usamos ciudad actual: compras históricas se reclasifican. SCD2 sería necesario para ciudad en fecha de compra.

### Solución 11

In [ ]:
feed=(spark.read.option("readChangeFeed","true").option("startingVersion",CDF_START).table("silver_customers"))
feed.write.format("delta").mode("overwrite").saveAsTable("customer_cdc_audit")
display(feed.groupBy("_change_type").count())
assert feed.count()==5  # Dos actualizaciones (4 imágenes) y un insert.

### Solución 12

In [ ]:
EVENT_SCHEMA="event_id INT, customer_id INT, event_type STRING, event_time TIMESTAMP"
EVENT_CHECKPOINT=BASE+"/checkpoints/exam_events"
def ingest_events():
    q=(spark.readStream.schema(EVENT_SCHEMA).json(LANDING).writeStream.format("delta")
       .option("checkpointLocation",EVENT_CHECKPOINT).trigger(availableNow=True).toTable("bronze_events"))
    q.awaitTermination()
    return q
ingest_events()
assert spark.table("bronze_events").count()==12

### Solución 13

In [ ]:
new_events=[{"event_id":i,"customer_id":1,"event_type":"purchase","event_time":"2026-01-10T11:00:00Z"} for i in range(13,16)]
dbutils.fs.put(LANDING+"/events_02.json","\n".join(json.dumps(e) for e in new_events),overwrite=False)
ingest_events()
ingest_events()
assert spark.table("bronze_events").count()==15

### Solución 14

In [ ]:
spark.table("bronze_events").filter("event_id IS NOT NULL AND event_time IS NOT NULL").dropDuplicates(["event_id"]).write.format("delta").mode("overwrite").saveAsTable("silver_events")
spark.sql("CREATE OR REPLACE TABLE gold_event_types USING DELTA AS SELECT event_type,COUNT(*) events FROM silver_events GROUP BY event_type")
assert spark.table("gold_event_types").agg(F.sum("events")).first()[0]==15

### Solución 15

In [ ]:
orders=spark.table("silver_orders")
assert orders.groupBy("order_id").count().filter("count>1").count()==0
assert orders.filter("quantity<=0 OR quantity IS NULL").count()==0
assert spark.table("bronze_orders").count()==orders.count()+1+spark.table("quarantine_invalid_orders").count()+spark.table("quarantine_orphan_orders").count()
expected_revenue=sum((i%3+1)*(((i-1)%6+1)*10) for i in range(1,25))
assert spark.table("gold_daily_city").agg(F.sum("revenue")).first()[0]==expected_revenue
print("Controles correctos; ingreso esperado:",expected_revenue)

### Solución 16

In [ ]:
spark.sql("CREATE OR REPLACE VIEW ecommerce_report AS SELECT * FROM gold_daily_city")
GROUP="dea_readers"
APPLY_GRANTS=False
for command in [f"GRANT USE CATALOG ON CATALOG {ident(CATALOG)} TO {ident(GROUP)}",f"GRANT USE SCHEMA ON SCHEMA {ident(CATALOG)}.{ident(SCHEMA)} TO {ident(GROUP)}",f"GRANT SELECT ON VIEW ecommerce_report TO {ident(GROUP)}"]:
    print(command)
    if APPLY_GRANTS:
        spark.sql(command)
display(spark.sql("SHOW GRANTS ON VIEW ecommerce_report"))

### Solución 17

In [ ]:
job_design={"bronze":[], "silver":["bronze"], "quality":["silver"], "gold":["quality"]}
print(json.dumps(job_design,indent=2))
# Divide las soluciones en notebooks de trabajo parametrizados con catalog/schema.
# Usa resources/bundle como referencia: retries y max_concurrent_runs=1.
# Calidad falla con assert antes de Gold; Repair run tras corregir la causa.
# Mide duración, estados downstream y numInputRows; Spark UI para shuffle/spill.
# No uses el bootstrap con UUID como tarea recurrente: el schema del job debe persistir.

### Solución 18

In [ ]:
for table in ["silver_customers","silver_orders","gold_daily_city","bronze_events"]:
    print(table,spark.table(table).count())
    display(spark.sql(f"DESCRIBE HISTORY {table}").select("version","operation").limit(5))
# SCD1 elegido: estado actual por cliente; no preserva ciudad histórica de compra.
# Para SCD2: secuencia + intervalos y join temporal como notebook 21.
# Limitación: Silver eventos se recalcula batch; Bronze sí es incremental.
# No hay conector empresarial real ni despliegue de job hasta ejecutar pasos correspondientes.
# La política CDC de este caso maneja upserts, no bajas: ampliar antes de usarla con DELETE.

## Respuestas razonadas

1. **B.** El checkpoint controla progreso de lectura, no identidad de negocio entre archivos.

2. **C.** SCD2 permite relacionar cada compra con el intervalo válido de la dimensión.

3. **A.** La dependencia de calidad debe bloquear publicación; repara cuando el contrato vuelva a cumplirse.